# Classic models (MIDI) — vodič kroz notebook

Isti zadatak kao u `classic_models_raw.ipynb` — prepoznavanje kompozitora po djelu — ali atributi se ovdje
izvlače iz MusicNet-ovih note-level (MIDI) anotacija (`train_labels/`, `test_labels/`), ne iz audio signala.
Nikakav `.wav` se ne učita, pa je ovaj pipeline mnogo brži od audio verzije.

Isti metodološki okvir: `work_id` grupisanje, `StratifiedGroupKFold`, work-level majority vote, macro F1
kao glavna metrika (dataset je neuravnotežen po kompozitorima).

In [1]:
%load_ext autoreload
%autoreload 2

# Ucitavanje podataka

## Učitavanje i pregled metapodataka

`metadata` je tabela u kojoj svaki red opisuje jedan audio zapis / stav: kompozitora, djelo, stav, ansambl,
trajanje itd. Naredne ćelije prvo provjeravaju nedostajuće vrijednosti i spisak kompozitora, a tek onda
odlučuju koje klase ostaju u eksperimentu.

Putanje `AUDIO_DIR` i `METADATA_PATH` su lokalne putanje sa računara na kojem je notebook pisan; na drugom
računaru moraju da pokazuju na odgovarajuću lokaciju MusicNet fajlova.


In [3]:
import numpy as np
import pandas as pd

from paths import AUDIO_DIR, METADATA_PATH

metadata = pd.read_csv(METADATA_PATH)

print(metadata.head())
print(metadata.columns)

     id  composer               composition                   movement  \
0  1727  Schubert  Piano Quintet in A major                 2. Andante   
1  1728  Schubert  Piano Quintet in A major         3. Scherzo: Presto   
2  1729  Schubert  Piano Quintet in A major  4. Andantino - Allegretto   
3  1730  Schubert  Piano Quintet in A major          5. Allegro giusto   
4  1733  Schubert   Piano Sonata in A major               2. Andantino   

        ensemble            source                      transcriber  \
0  Piano Quintet  European Archive  http://tirolmusic.blogspot.com/   
1  Piano Quintet  European Archive  http://tirolmusic.blogspot.com/   
2  Piano Quintet  European Archive  http://tirolmusic.blogspot.com/   
3  Piano Quintet  European Archive  http://tirolmusic.blogspot.com/   
4     Solo Piano          Museopen                Segundo G. Yogore   

  catalog_name  seconds  
0        OP114      447  
1        OP114      251  
2        OP114      444  
3        OP114      368 

In [4]:
metadata.isna().sum()
# No missing values

id              0
composer        0
composition     0
movement        0
ensemble        0
source          0
transcriber     0
catalog_name    0
seconds         0
dtype: int64

In [5]:
metadata["composer"].unique()
# We have 10 composers

<StringArray>
[ 'Schubert',    'Mozart',    'Dvorak',   'Cambini',     'Haydn',    'Brahms',
     'Faure',     'Ravel',      'Bach', 'Beethoven']
Length: 10, dtype: str

## Zašto se pravi `work_id`?

Jedno muzičko djelo može imati više stavova, a u metapodacima su oni posebni redovi. Kombinacijom
`composer + composition` pravi se identifikator cijelog djela. Tako npr. više stavova iste sonate ostaju ista
grupa tokom cross-validationa.

Nakon toga se broji broj **različitih djela po kompozitoru**, ne broj segmenata. Kompozitori sa manje od 5
djela se uklanjaju jer bi grupni cross-validation za njih bio vrlo nestabilan.


In [6]:
metadata["work_id"] = (
    metadata["composer"].astype(str)
    + " | "
    + metadata["composition"].astype(str)
)

# Adding a column so we can group the same composition from different movements

In [7]:
work_counts = (
    metadata.groupby("composer")["work_id"]
    .nunique()
)

print(work_counts)

# We will remove all the composers that have less than 5 compositions in the dataset, also we can see our dataset is inbalanced

composer
Bach         30
Beethoven    55
Brahms        8
Cambini       3
Dvorak        2
Faure         1
Haydn         1
Mozart       11
Ravel         1
Schubert      9
Name: work_id, dtype: int64


In [8]:
valid_composers = work_counts[
    work_counts >= 5
].index

metadata_filtered = metadata[
    metadata["composer"].isin(valid_composers)
].copy()

# Filtering the composers

In [9]:
print(
    metadata_filtered.groupby("composer")["work_id"]
    .nunique()
    .sort_values()
)

composer
Brahms        8
Schubert      9
Mozart       11
Bach         30
Beethoven    55
Name: work_id, dtype: int64


---

## Group-aware evaluacija

Želimo da mjerimo generalizaciju na **nova djela**, ne na nove segmente poznatog djela. `StratifiedGroupKFold`
drži sve segmente istog `work_id` u istom foldu i približno čuva raspodjelu kompozitora po foldovima.

In [ ]:
from sklearn import metrics
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier

from evaluation import make_sample_weights, majority_vote_prediction

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

---

# MIDI model

Svaka nota u MusicNet anotacijama ima visinu, instrument, tačan onset/offset i trajanje u otkucajima.
`extract_midi_features` (u `midi.py`) iz toga računa gustinu nota, melodijski kontur (interval mean/std,
udio uzlaznih/silaznih/ponovljenih koraka), histogram tonskih klasa, histogram instrumenata (11 fiksnih GM
programa u MusicNet-u), on-beat/off-beat omjer i raspon akorada.

Napomena: `end_beat` kolona u MusicNet CSV-u je zapravo **trajanje note u otkucajima**, ne apsolutna
pozicija (provjereno — `end_beat < start_beat` za svaki red u fajlu).

In [ ]:
from midi import make_midi_dataset

X_midi, y_midi, groups_midi = make_midi_dataset(metadata_filtered, AUDIO_DIR)

print(X_midi.shape)

In [ ]:
all_work_true_midi = []
all_work_pred_midi = []

for train_idx, val_idx in sgkf.split(X_midi, y_midi, groups_midi):
    X_train_fold = X_midi[train_idx]
    y_train_fold = y_midi[train_idx]
    groups_train = groups_midi[train_idx]

    X_val_fold = X_midi[val_idx]
    y_val_fold = y_midi[val_idx]
    groups_val = groups_midi[val_idx]

    sample_weights = make_sample_weights(y_train_fold, groups_train)

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_fold, y_train_fold, sample_weight=sample_weights)

    pred = model.predict(X_val_fold)

    work_true, work_pred = majority_vote_prediction(y_val_fold, pred, groups_val)

    all_work_true_midi.extend(work_true)
    all_work_pred_midi.extend(work_pred)

print(metrics.classification_report(all_work_true_midi, all_work_pred_midi))

---

## XGBoost

Gradient boosting ansambl — stabla se dodaju sekvencijalno, svako ispravlja greške prethodnih. Često
nadmaši Random Forest na ovakvim tabelarnim feature setovima. `LabelEncoder` je potreban jer XGBoost
očekuje numeričke labele, ne stringove.

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(y_midi)

all_work_true = []
all_work_pred = []

for train_idx, val_idx in sgkf.split(X_midi, y_midi, groups_midi):
    X_train_fold = X_midi[train_idx]
    y_train_fold = le.transform(y_midi[train_idx])
    groups_train = groups_midi[train_idx]

    X_val_fold = X_midi[val_idx]
    y_val_fold = y_midi[val_idx]
    groups_val = groups_midi[val_idx]

    sample_weights = make_sample_weights(y_midi[train_idx], groups_train)

    model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.1,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_fold, y_train_fold, sample_weight=sample_weights)

    pred = le.inverse_transform(model.predict(X_val_fold))

    work_true, work_pred = majority_vote_prediction(y_val_fold, pred, groups_val)

    all_work_true.extend(work_true)
    all_work_pred.extend(work_pred)

print(metrics.classification_report(all_work_true, all_work_pred))